<h2 style='color:green' align="center">Intelligent intent detection for AI-Defense (Kuantum Lestari)</h2>

  
## FULL DEVELOPMENT PIPELINE  🏗️ ##
### 1. Install dependencies ###

In [1]:
!pip install psycopg2-binary pgvector ollama


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\amikf\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


### 2. Load your dataset ###

In [3]:
import pandas as pd

df = pd.read_csv("geopolitics_intent_dataset.csv")

texts = df["text"].tolist()
labels = df["intent"].tolist()

About the dataset:

This dataset consists of 90 text samples designed for natural language processing (NLP) tasks, particularly intent classification in the domain of geopolitics and conflict-related queries. It contains four main variables: text, intent, language, and paraphrase_group. The text column represents user queries about geopolitical issues such as war updates, conflicts, and international tensions. The intent column is the dependent variable, labeling each query into categories like “war_update,” which indicates the user’s informational goal. The language column shows that the dataset is multilingual, including English (en), Malay (ms), and mixed-language (mix), making it suitable for multilingual semantic modeling. The paraphrase_group groups semantically similar queries together, meaning different phrasings of the same question are clustered, which is useful for training models to understand variations in human language.

### 3. Visualize dataset ###

This visualization  df.head()  shows the first five rows of the dataset, giving a quick look at its structure.

In [4]:
df.head()

,text,intent,language,paraphrase_group
0,Is there any war happening right now?,war_update,en,1
1,Ada perang sekarang ke?,war_update,ms,1
2,Any latest conflict news sekarang?,war_update,mix,1
3,Which countries are in conflict currently?,war_update,en,2
4,Negara mana tengah berperang sekarang?,war_update,ms,2


In [5]:
df.shape

(90, 4)

### 4. Modelling ###
- Connect PostgresDB + Ollama Embeddings
- Generate embeddings
- Choose embedding model (IMPORTANT)

        Use multilingual (Malay + English):

- use nomic-embed-text as the AI model

In [6]:
import psycopg2
from pgvector.psycopg2 import register_vector
from ollama import embeddings

# PostgreSQL connection
conn = psycopg2.connect(
    host="localhost",
    database="aid",
    user="postgres",
    password="password"
)

cur = conn.cursor()

# Enable pgvector
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Create table
cur.execute("""
CREATE TABLE IF NOT EXISTS intents (
    text TEXT,
    intent TEXT,
    embedding VECTOR(768)
)
""")

conn.commit()

We sets up a system to convert text into numerical embeddings and store them in a vector database using PostgresDB. It imports necessary modules, then defines an embedding function using Ollama with the “nomic-embed-text” model

Next, creates a persistent database stored in the ./chroma_db folder, meaning the data will be saved even after restarting the program.

Finally, creates or loads a collection called "intent_db" and connects it with the embedding function, so any text added to this collection will automatically be converted into embeddings for tasks like semantic search or intent matching.

 ### 5. Store in Vector DB (PGVector) ###

Now, we takes your dataset and saves it into the PG vector database.

It first creates simple IDs for each text, then stores the texts along with their intent labels as metadata. When saved, each text is automatically converted into embeddings, so you can later search by meaning, not just exact words.

In [7]:
for text, label in zip(texts, labels):


    # Generate embedding using Ollama
    response = embeddings(
        model='nomic-embed-text',
        prompt=text
    )

    embedding = response["embedding"]

    # Insert into PostgreSQL
    cur.execute("""
        INSERT INTO intents
        (text, intent, embedding)
        VALUES ( %s, %s, %s)
    """, (text, label, embedding))

conn.commit()

### 6. Query pipeline for PostgresDB (Intent detection) ###

k is the number of nearest results (neighbors) to retrieve from the vector database.

When k=3, the system looks at the top 3 most similar texts to your query, and then uses the closest one (or compares them) to decide the predicted intent.

In [45]:
def predict_intent(query, k=3):

    # Convert query into embedding
    response = embeddings(
        model='nomic-embed-text',
        prompt=query
    )

    query_embedding = response["embedding"]

    # Vector similarity search
    cur.execute("""
        SELECT text,
               intent,
               embedding <-> %s::vector AS distance
        FROM intents
        ORDER BY distance
        LIMIT %s
    """, (query_embedding, k))

    results = cur.fetchall()

    best_text = results[0][0]
    best_intent = results[0][1]
    best_distance = results[0][2]

    best_score = 1 - best_distance

    return best_intent, best_score


# Example test


#query = "Iran war update sekarang"
#intent, score = predict_intent(query)

#print(intent, score)

We predicts the intent of a given query by searching similar texts in the PostgreDB collection

It sends the query to the database, retrieves the top k most similar results, and extracts their documents, metadata, and distances.

The intent of the closest match is selected as the predicted intent, and its similarity score is calculated by converting distance into similarity (1 - distance).

### 7. Evaluation ###

y_true as the actual correct labels (ground truth), and creates an empty list y_pred to store the model’s predicted labels later.

The threshold = 0.75 is used to decide how confident a prediction must be to be accepted

In [46]:
from sklearn.metrics import classification_report

y_true = labels
y_pred = []

threshold = 0.75

### 8. Inference (Prediction Phase) ###

In [47]:
for text in texts:
    intent, score = predict_intent(text)

    if score >= threshold:
        y_pred.append(intent)
    else:
        y_pred.append("unknown")

We goes through each text in the dataset and uses a function (predict_intent) to predict its intent along with a confidence score

It adds the predicted intent to y_pred only if the score meets the threshold, otherwise labeling it as "unknown".

This helps ensure that only reliable predictions are accepted, while uncertain ones are filtered out.

### 9. Accuracy Model (F1-Score) ###

In [48]:
print(classification_report(y_true, y_pred))

                      precision    recall  f1-score   support

       defense_alert       1.00      1.00      1.00         9
geopolitics_analysis       1.00      1.00      1.00         9
    iran_geopolitics       1.00      1.00      1.00         9
         iran_hormuz       1.00      1.00      1.00         9
       iran_military       1.00      1.00      1.00         9
           iran_risk       1.00      1.00      1.00         9
     iran_war_update       1.00      1.00      1.00         9
       military_news       1.00      1.00      1.00         9
     risk_assessment       1.00      1.00      1.00         9
          war_update       1.00      1.00      1.00         9

            accuracy                           1.00        90
           macro avg       1.00      1.00      1.00        90
        weighted avg       1.00      1.00      1.00        90



For every class (like war_update, military_news, etc.), the precision, recall, and F1-score are all 1.00, meaning the model made perfectly correct predictions with no mistakes

The support value (9 for each class) indicates how many samples were tested per category, totaling 90 samples overall.

The overall accuracy is 1.00 (100%), and both macro and weighted averages are also perfect, showing that the model performed consistently well across all classes without bias or imbalance issues.

### 8. OPTIONAL: Use GEMMA for final response (LLM layer) ###

In [ ]:
import requests

def ask_gemma(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:1b",
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]